In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib

print('✅ Setup complete')

✅ Setup complete


In [2]:
# Load all fitted artifacts from training
label_encoders = joblib.load('../models/label_encoders.pkl')
scaler = joblib.load('../models/scaler.pkl')
polynomial = joblib.load('../models/polynomial.pkl')
selected_features = joblib.load('../models/selected_features.pkl')
model = joblib.load('../models/xgb_churn.pkl')

print('✅ Artifacts loaded')
print(f'   Selected features: {len(selected_features)}')

✅ Artifacts loaded
   Selected features: 20


In [3]:
# One illustrative new customer
new_customer = {
    'credit_score': 650,
    'country': 'France',
    'gender': 'Female',
    'age': 42,
    'tenure': 3,
    'balance': 125000.00,
    'products_number': 1,
    'credit_card': 1,
    'active_member': 0,
    'estimated_salary': 95000.00
}

new_customer_df = pd.DataFrame([new_customer])
new_customer_df

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,650,France,Female,42,3,125000.0,1,1,0,95000.0


In [4]:
# Apply the same label encoders used in training
for column_name, encoder in label_encoders.items():
    new_customer_df[column_name] = encoder.transform(new_customer_df[column_name])

new_customer_df

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,650,0,0,42,3,125000.0,1,1,0,95000.0


In [5]:
# Apply the same scaler used in training
encoded_columns = list(label_encoders.keys())
numerical_columns = new_customer_df.select_dtypes(
    include=['int64', 'float64']
).columns.difference(encoded_columns)

new_customer_df[numerical_columns] = scaler.transform(
    new_customer_df[numerical_columns]
)

new_customer_df

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary
0,-0.005471,0,0,0.293517,-0.695982,0.777541,-0.911583,0.646092,-1.03067,-0.088514


In [6]:
# Apply the same polynomial transformer used in training
all_numerical_columns = new_customer_df.select_dtypes(
    include=['int64', 'float64']
).columns

polynomial_array = polynomial.transform(new_customer_df[all_numerical_columns])
polynomial_feature_names = polynomial.get_feature_names_out(all_numerical_columns)

polynomial_df = pd.DataFrame(
    polynomial_array,
    columns=polynomial_feature_names,
    index=new_customer_df.index
)

new_feature_columns = [
    column for column in polynomial_df.columns
    if column not in new_customer_df.columns
]

full_features_df = pd.concat(
    [new_customer_df, polynomial_df[new_feature_columns]],
    axis=1
)

print(f'✅ {len(new_feature_columns)} polynomial features created')

✅ 55 polynomial features created


In [7]:
# Filter to only the features selected during training
final_input = full_features_df[selected_features]
final_input

,age,products_number^2,products_number active_member,balance products_number,age active_member,age products_number,country age,gender age,age balance,products_number,balance^2,balance,estimated_salary^2,balance active_member,tenure estimated_salary,balance estimated_salary,credit_score balance,products_number estimated_salary,credit_card estimated_salary,country balance
0,0.293517,0.830984,0.939542,-0.708793,-0.30252,-0.267566,0.0,0.0,0.228222,-0.911583,0.60457,0.777541,0.007835,-0.801388,0.061604,-0.068823,-0.004254,0.080688,-0.057188,0.0


In [8]:
# Generate churn probability
churn_probability = model.predict_proba(final_input)[0][1]

print(f'\nCHURN PREDICTION')
print(f'{"=" * 40}')
print(f'Churn probability: {churn_probability:.2%}')

if churn_probability > 0.7:
    print('Risk segment: HIGH')
elif churn_probability > 0.4:
    print('Risk segment: MEDIUM')
else:
    print('Risk segment: LOW')


CHURN PREDICTION
Churn probability: 58.69%
Risk segment: MEDIUM
